In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

def find_archive():
    for base in ['E:/archive', '../data/row/archive', '/kaggle/input']:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            if 'Data_Entry_2017.csv' in files:
                return root
    raise FileNotFoundError('Data_Entry_2017.csv not found — set the path manually.')

ARCHIVE = find_archive()
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else '../data/processed'
os.makedirs(OUT, exist_ok=True)
print('archive:', ARCHIVE, '\noutput :', OUT)

LABELS = ['Atelectasis','Cardiomegaly','Consolidation','Edema','Effusion','Emphysema',
          'Fibrosis','Hernia','Infiltration','Mass','Nodule','Pleural_Thickening',
          'Pneumonia','Pneumothorax']

# knobs (match src/preprocess.py defaults)
VAL_FRAC, NF_RATIO, POS_CAP, SEED = 0.10, 0.5, 10.0, 42
rng = np.random.default_rng(SEED)

df = pd.read_csv(os.path.join(ARCHIVE, 'Data_Entry_2017.csv'))
print('raw shape:', df.shape)

## Clean impossible ages

In [ ]:
before = len(df)
df = df[df['Patient Age'] <= 100].copy()
print(f'dropped {before - len(df)} rows with age > 100  ->  {len(df):,} remain')

## Encode gender & normalize age

In [ ]:
df['Patient Gender'] = (df['Patient Gender'] == 'F').astype('float32') 
df['Patient Age'] = df['Patient Age'].astype('float32') / 100.0
df[['Patient Age', 'Patient Gender']].describe().round(3)

## Expand the 14 binary label columns

Turn the pipe-separated `Finding Labels` string into 14 numeric columns the Dataset can read directly (no string parsing at train time).

In [ ]:
for L in LABELS:
    df[L] = df['Finding Labels'].fillna('').str.contains(L, regex=False).astype('float32')
df = df[['Image Index', 'Patient ID', 'Patient Age', 'Patient Gender'] + LABELS]
df.head(3)

## Step 4 — Official split + patient-wise validation

Use NIH's official `train_val_list.txt` / `test_list.txt`, then carve validation **by patient** so no patient appears in both train and val.

In [ ]:
tv = set(open(os.path.join(ARCHIVE, 'train_val_list.txt')).read().split())
te = set(open(os.path.join(ARCHIVE, 'test_list.txt')).read().split())
trainval = df[df['Image Index'].isin(tv)].copy()
test     = df[df['Image Index'].isin(te)].copy()

patients = trainval['Patient ID'].unique(); rng.shuffle(patients)
val_patients = set(patients[:int(len(patients) * VAL_FRAC)])
val   = trainval[trainval['Patient ID'].isin(val_patients)].copy()
train = trainval[~trainval['Patient ID'].isin(val_patients)].copy()

overlap = set(train['Patient ID']) & set(val['Patient ID'])
print(f'train {len(train):,} | val {len(val):,} | test {len(test):,}')
print(f'patient overlap train∩val: {len(overlap)}  (must be 0)')
assert len(overlap) == 0

## Undersample No-Finding (train only)

Keep every image with ≥1 finding; keep at most `NF_RATIO ×` that many healthy images. **Val/test are untouched** — they must reflect the real-world distribution for honest evaluation.

In [ ]:
has = train[LABELS].sum(axis=1) > 0
pos, neg = train[has], train[~has]
keep = int(len(pos) * NF_RATIO)
neg_bal = neg.sample(n=keep, random_state=SEED) if keep < len(neg) else neg
train_bal = pd.concat([pos, neg_bal]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

h = lambda d: (d[LABELS].sum(axis=1) == 0).mean()
print(f'train healthy share: {h(train):.1%} (raw) -> {h(train_bal):.1%} (balanced)')

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for a, (name, frac) in zip(ax, [('raw train', h(train)), ('balanced train', h(train_bal))]):
    a.pie([frac, 1 - frac], labels=['No Finding', 'has findings'],
          autopct='%1.0f%%', colors=['crimson', 'steelblue'], startangle=90)
    a.set_title(f'{name}  (n={len(train) if name=="raw train" else len(train_bal):,})')
plt.tight_layout(); plt.show()

## Save and compute pos_weight

`pos_weight = neg/pos` per class (capped) handles the residual rare-disease imbalance inside `BCEWithLogitsLoss`. `train.py` recomputes this automatically — shown here for transparency.

In [ ]:
for name, d in [('train', train_bal), ('val', val), ('test', test)]:
    d.drop(columns=['Patient ID']).to_csv(f'{OUT}/{name}.csv', index=False)
print('wrote train.csv / val.csv / test.csv ->', OUT)

pw = {L: min((len(train_bal) - train_bal[L].sum()) / max(train_bal[L].sum(), 1), POS_CAP) for L in LABELS}
pos_tbl = pd.DataFrame({'positives': [int(train_bal[L].sum()) for L in LABELS],
                        'pos_weight': [round(pw[L], 2) for L in LABELS]}, index=LABELS)
print(pos_tbl.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
pos_tbl['pos_weight'].plot.barh(ax=ax, color='darkcyan')
ax.axvline(POS_CAP, ls='--', c='red', label=f'cap = {POS_CAP:.0f}')
ax.set_xlabel('pos_weight'); ax.set_title('Per-class pos_weight (capped)')
ax.legend(); ax.invert_yaxis(); plt.tight_layout(); plt.show()

## Sanity check the output

Reload `train.csv` and confirm a row has the expected normalized clinical features and a valid label vector.

In [ ]:
check = pd.read_csv(f'{OUT}/train.csv')
row = check.iloc[0]
print('Image      :', row['Image Index'])
print('Age (norm) :', row['Patient Age'], ' Gender:', row['Patient Gender'])
print('Findings   :', [L for L in LABELS if row[L] == 1] or ['No Finding'])
print('\nAge in [0,1]?', check['Patient Age'].between(0, 1).all())
print('Gender in {0,1}?', check['Patient Gender'].isin([0, 1]).all())
print('\nReady for src/train.py ✓')